# Notebook 08: Gold/Evaluation Set for Skill Extraction

- Create a small, qualitative gold set (sample) from external datasets + export Excel template
- Skills are manually noted for each document (in `skills_manual`)
- The gold set will later serve as a reference for quantitative evaluation (e.g., precision/recall, etc.) and comparison with the results of the profiler extension methods
- Import the completed gold set & prepare labels
- Load predictions from the methods (e.g., 1.1 rule-based, 2.1 text mining) and compare them against the gold set
- This notebook is created at this stage before the profile expansion is performed, as it is prepared in advance. After the profile expansions are carried out (NB 10-11), this notebook will be used.

Theory:
- Manually annotated gold datasets are the standard for evaluating skill extraction. Models are evaluated against manually curated gold sets, and various metrics—including ranking metrics (depending on the setup)—are used (Decorte et al. 2023; Cenikj et al. 2021; Shi et al. 2020)
- Ranking metrics such as MRR or R-Precision@k are frequently used in skill ranking or recommender setups (Aggarwal, 2016; Decorte et al., 2023). However, since the methods considered here generate only unordered skill sets without scores, we limit ourselves to set-based metrics (Precision/Recall/F1).
- Job recommender approaches also use expert-labeled sets as ground truth (Gugnani et al. 2020; Shi et al. 2020)
- Our approach follows this logic on a smaller scale: creating a gold set of 25 job postings as a reference
(Additional test notebook available at notebooks/08b_gold_eval_set_EN_only/)

## 1. Create a template

In [9]:
# Setup + Paths
from pathlib import Path
import pandas as pd

# Project Root
PROJECT_ROOT = Path(".").resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Paths
DATA = PROJECT_ROOT / "data"
DATA_PROCESSED_EXTERNAL = DATA / "processed_external"
SRC_PATH = DATA_PROCESSED_EXTERNAL / "documents_raw.parquet"
LIGHT_PATH = DATA_PROCESSED_EXTERNAL / "documents_raw_light.parquet"
GOLD_IDS_PATH = DATA_PROCESSED_EXTERNAL / "gold_doc_ids.csv"
GOLD_TEMPLATE_XLSX = DATA_PROCESSED_EXTERNAL / "goldset_for_annotation.xlsx"
GOLD_LABELED_XLSX = DATA_PROCESSED_EXTERNAL / "goldset_labeled.xlsx"

# Predictions from the two methods
PRED_RULEBASED_11 = DATA_PROCESSED_EXTERNAL / "pred_skills_10b_rulebased.parquet"
PRED_MINING_21 = DATA_PROCESSED_EXTERNAL / "method_2_1" / "pred_skills_11_mining_21.parquet"
PRED_FILES = {"rule_based_1.1": PRED_RULEBASED_11, "method_2.1_text_mining": PRED_MINING_21,}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LIGHT_PATH exists:", LIGHT_PATH.exists(), LIGHT_PATH)
print("GOLD_IDS_PATH exists:", GOLD_IDS_PATH.exists(), GOLD_IDS_PATH)
print("GOLD_TEMPLATE_XLSX exists:", GOLD_TEMPLATE_XLSX.exists(), GOLD_TEMPLATE_XLSX)
print("GOLD_LABELED_XLSX exists:", GOLD_LABELED_XLSX.exists(), GOLD_LABELED_XLSX)
for k, p in PRED_FILES.items():
    print(f"PRED {k} exists:", p.exists(), p)

# external datasets for Gold/Eval; “course” is excluded; optionally: “kaggle_linkedin_morocco”
KEEP_SOURCES = ["workwise_selenium","ba_jobsuche_api_2025","kaggle_linkedin_2023_2024_big","kaggle_techsalerator","kaggle_resume_structured",]

PROJECT_ROOT: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
LIGHT_PATH exists: True C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\documents_raw_light.parquet
GOLD_IDS_PATH exists: False C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\gold_doc_ids.csv
GOLD_TEMPLATE_XLSX exists: True C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\goldset_for_annotation.xlsx
GOLD_LABELED_XLSX exists: True C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\goldset_labeled.xlsx
PRED rule_based_1.1 exists: True C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\pred_skills_10b_rulebased.parquet
PRED method_2.1_text_mining exists: True C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_ex

### 1.1 Light Copy

Starting point: documents_raw.parquet. Loading the entire `documents_raw.parquet` table can sometimes cause memory issues, so a light copy is created (excluding udemy and with truncated columns):
- smaller batches
- no batch.to_table()
- Light file with preview text (8,000 characters)

In [10]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pyarrow.compute as pc
from pathlib import Path
import os

# Input = SRC_PATH, Output = LIGHT_PATH
print("SRC_PATH:", SRC_PATH)
print("LIGHT_PATH:", LIGHT_PATH)
COLS_NEEDED = ["doc_id","source_type","source_name","job_title_raw","raw_text","language","meta_json"] # Spalten behalten

filter_expr = ds.field("source_name").isin(KEEP_SOURCES) if "KEEP_SOURCES" in globals() else None # Filter

MAX_CHARS = 8000 # Preview length
BATCH_SIZE = 64 # more stable

LIGHT_TMP = Path(str(LIGHT_PATH) + ".tmp") # first tmp, then rename
if LIGHT_TMP.exists():
    LIGHT_TMP.unlink()

dataset = ds.dataset(SRC_PATH, format="parquet")  # Input
scanner = dataset.scanner(filter=filter_expr,columns=COLS_NEEDED,batch_size=BATCH_SIZE,use_threads=False)
target_schema = pa.schema([pa.field("doc_id", pa.string()),pa.field("source_type", pa.string()),pa.field("source_name", pa.string()),pa.field("job_title_raw", pa.string()),pa.field("raw_text", pa.string()), pa.field("language", pa.string()),pa.field("meta_json", pa.string()),])
writer = None
rows_written = 0

for rb in scanner.to_batches():
    tbl = pa.Table.from_batches([rb])
    raw = tbl["raw_text"] # raw_text as ""
    raw = pc.if_else(pc.is_null(raw), pa.scalar("", pa.string()), raw)
    raw = pc.utf8_slice_codeunits(raw, start=0, stop=MAX_CHARS)
    tbl = tbl.set_column(tbl.schema.get_field_index("raw_text"), "raw_text", raw)
    tbl = tbl.set_column(tbl.schema.get_field_index("meta_json"), "meta_json", pc.cast(tbl["meta_json"], pa.string(), safe=False)) # Cast meta_json to a string
    tbl = tbl.select(target_schema.names).cast(target_schema, safe=False) # Schema, Order  + Types

    if writer is None:
        writer = pq.ParquetWriter(LIGHT_TMP, target_schema, compression="snappy")

    writer.write_table(tbl)
    rows_written += tbl.num_rows

if writer is not None:
    writer.close()

# tmp final
if LIGHT_PATH.exists():
    LIGHT_PATH.unlink()
LIGHT_TMP.rename(LIGHT_PATH)

print("Fertig. Zeilen:", rows_written)
print("Gespeichert:", LIGHT_PATH)

SRC_PATH: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\documents_raw.parquet
LIGHT_PATH: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\documents_raw_light.parquet
Fertig. Zeilen: 192480
Gespeichert: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\documents_raw_light.parquet


### 1.2 Loading from Light Doc

Contains all text sources in the Unified Document Schema: `doc_id`, `source_type`, `source_name`, `job_title_raw`, `raw_text` (truncated), `language`, `meta_json`

In [21]:
df_docs = pd.read_parquet(LIGHT_PATH, engine="pyarrow")
print("Anzahl Dokumente (light):", len(df_docs))
display(df_docs[["source_type","source_name"]].value_counts().head(20))
display(df_docs["language"].value_counts(dropna=False).head(20))
df_docs.head(3)

Anzahl Dokumente (light): 192480


source_type    source_name                  
job_ad         kaggle_linkedin_2023_2024_big    123849
cv             kaggle_resume_structured          54933
job_ad         kaggle_techsalerator               9807
               ba_jobsuche_api_2025               3505
job_ad_scrape  workwise_selenium                   386
Name: count, dtype: int64

language
en       185846
de         5131
pt          535
es          274
fr          172
pl          115
cs           95
nl           87
sl           68
hu           47
ro           35
sk           18
tr           15
da           14
ja           12
ru            4
it            3
uk            3
zh-cn         2
vi            2
Name: count, dtype: int64

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,Marketing Coordinator\n\nJob descriptionA lead...,en,"{""job_id"": 921716, ""company_id"": 2774458.0, ""l..."
1,1829192,job_ad,kaggle_linkedin_2023_2024_big,Mental Health Therapist/Counselor,Mental Health Therapist/Counselor\n\nAt Aspen ...,en,"{""job_id"": 1829192, ""company_id"": NaN, ""locati..."
2,10998357,job_ad,kaggle_linkedin_2023_2024_big,Assitant Restaurant Manager,Assitant Restaurant Manager\n\nThe National Ex...,en,"{""job_id"": 10998357, ""company_id"": 64896719.0,..."


Distribution of external data records by source_type. A total of 192,480 entries. EN dominates (LinkedIn resumes and job ads), while DE accounts for a smaller share (BA and Workwise).

### 1.3 Coverage Check: Datasets with Skill Extraction

- df_docs contains all documents from the sources
- Problem: In your methods (10b rule_based_1.1, 11 method_2.1), skills were extracted only for documents that had been previously mapped or processed (due to RAM constraints)
- Therefore, the gold set must come from documents that actually appear in the prediction files (pred_skills_*.parquet); otherwise, the subsequent evaluation will be unfair or empty.

Procedure: Mark the following in df_docs for each document:
- has_rulebased = doc_id appears in pred_skills_10b_rulebased.parquet
- has_mining = doc_id appears in pred_skills_11_mining_21.parquet
- has_both = both True -> select only records from this. The gold set is explicitly drawn from this intersection.

In [22]:
# doc_id: Intersection between Light-Docs and pred_skills
from tqdm import tqdm

def collect_unique_doc_ids_parquet(parquet_path, col="doc_id", max_row_groups=None): # Read only the (doc_id) column by rowgroup and collect unique values into a set, max_row_groups
    pf = pq.ParquetFile(parquet_path)
    n_rg = pf.num_row_groups # in row_groups, so it's more stable than loading Pandas
    if max_row_groups is not None:
        n_rg = min(n_rg, max_row_groups)

    out = set()
    for rg in tqdm(range(n_rg), desc=f"scan doc_ids: {parquet_path.name}", leave=False):
        tbl = pf.read_row_group(rg, columns=[col])
        arr = tbl[col].to_pylist()
        out.update([x for x in arr if x is not None]) # filter None 
    return out

# doc_id sets from prediction files
rule_doc_ids = collect_unique_doc_ids_parquet(PRED_RULEBASED_11, col="doc_id")
mining_doc_ids = collect_unique_doc_ids_parquet(PRED_MINING_21, col="doc_id")
print("unique docs rule_based:", len(rule_doc_ids))
print("unique docs text_mining:", len(mining_doc_ids))

# In df_docs, indicate whether the document appears in the methods
df_docs["has_rulebased"] = df_docs["doc_id"].isin(rule_doc_ids)
df_docs["has_mining"] = df_docs["doc_id"].isin(mining_doc_ids)
df_docs["has_both"] = df_docs["has_rulebased"] & df_docs["has_mining"]

# Overview by source
cov = (df_docs.groupby(["source_name", "language"], dropna=False).agg(n_docs=("doc_id", "count"),n_rule=("has_rulebased", "sum"),n_mining=("has_mining", "sum"),n_both=("has_both", "sum"),)
    .sort_values(["n_both", "n_docs"], ascending=False))

display(cov.head(50))
print("\nTotal docs in light:", len(df_docs))
print("Docs mit rulebased preds:", int(df_docs["has_rulebased"].sum()))
print("Docs mit mining preds:", int(df_docs["has_mining"].sum()))
print("Docs mit beiden preds:", int(df_docs["has_both"].sum()))

unique docs rule_based: 187970
unique docs text_mining: 62161


n_docs  n_rule  n_mining  n_both
source_name                   language                                  
kaggle_resume_structured      en         54933   54060     31350   31341
kaggle_linkedin_2023_2024_big en        123849  119614     27847   27842
kaggle_techsalerator          en          7064    6785      1629    1629
ba_jobsuche_api_2025          de          3505    3278       631     630
kaggle_techsalerator          de          1240    1082       138     138
                              pt           535     432       105     105
                              es           274     255        38      38
                              fr           172     153        19      19
                              cs            95      90        16      16
                              nl            87      79        13      13
                              hu            47      39        11      11
                              ja            12      11         9       9
                              ro            35      25         8       8
                              da            14      14         7       7
                              pl           115      83         5       5
                              sk            18      18         4       4
                              vi             2       2         2       2
                              tr            15      13         1       1
workwise_selenium             de           386       0         0       0
kaggle_techsalerator          sl            68      32         0       0
                              ru             4       0         0       0
                              it             3       2         0       0
                              uk             3       3         0       0
                              zh-cn          2       1         0       0
                              bg             1       0         0       0
                              et             1       1         0       0


Total docs in light: 192480
Docs mit rulebased preds: 186072
Docs mit mining preds: 61833
Docs mit beiden preds: 61818


61,910 external data points match across both profile expansion methods, meaning they were used in both for skill extraction.

### 1.4 Selecting the Gold Set

Select a sample (gold set): a small, heterogeneous gold set for later evaluation of skill extraction. Select the documents such that skill extraction has definitely been performed on them.
- Selection via random sampling (`sample(..., random_state=42)`) based on `documents_raw_light.parquet`
- 5 German-language Workwise ads (`source_name=“workwise_selenium”`, `language=“de”`)
- 5 German-language BA job ads (`source_name=“ba_jobsuche_api_2025”`, `language=“de”`)
- 10 English job ads (primarily `source_name=‘kaggle_linkedin_2023_2024_big’`, otherwise `kaggle_techsalerator`, each with `language=“en”`)
- 5 CV documents (`source_name=“kaggle_resume_structured”`)

In [23]:
RANDOM_STATE = 42
# N_WORKWISE_DE = 5
N_WORKWISE_DE = 0
N_BA_DE = 5
N_EN_JOBS = 10
N_CV = 5

def safe_sample(df, mask, n, label, random_state=42):
    sub = df[mask].copy()
    if len(sub) < n:
        raise ValueError(f"Nicht genug Daten für {label}: brauche {n}, habe {len(sub)}")
    return sub.sample(n=n, random_state=random_state)

# only docs with BOTH
df_docs_both = df_docs[df_docs["has_both"]].copy()
print("Datensätze die fürs sample in Frage kommen (has_both=True):", len(df_docs_both))

# Workwise (DE)
# mask_workwise = (df_docs_both["source_name"] == "workwise_selenium") & (df_docs_both["language"] == "de")
# workwise_sample = safe_sample(df_docs_both, mask_workwise, N_WORKWISE_DE, "workwise_selenium (DE)", random_state=RANDOM_STATE)
workwise_sample = df_docs_both.iloc[0:0].copy()

# BA (DE)
mask_ba = (df_docs_both["source_name"] == "ba_jobsuche_api_2025") & (df_docs_both["language"] == "de")
ba_sample = safe_sample(df_docs_both, mask_ba, N_BA_DE, "ba_jobsuche_api_2025 (DE)", random_state=RANDOM_STATE)

# EN Job Ads (primarily LinkedIn big, fallback techsalator)
if (df_docs_both["source_name"] == "kaggle_linkedin_2023_2024_big").any():
    mask_en = (df_docs_both["source_name"] == "kaggle_linkedin_2023_2024_big") & (df_docs_both["language"] == "en")
    en_sample = safe_sample(df_docs_both, mask_en, N_EN_JOBS, "kaggle_linkedin_2023_2024_big (EN)", random_state=RANDOM_STATE)
else:
    mask_en = (df_docs_both["source_name"] == "kaggle_techsalator") & (df_docs_both["language"] == "en")
    en_sample = safe_sample(df_docs_both, mask_en, N_EN_JOBS, "kaggle_techsalator (EN fallback)", random_state=RANDOM_STATE)

# CVs
mask_cv = (df_docs_both["source_name"] == "kaggle_resume_structured")
cv_sample = safe_sample(df_docs_both, mask_cv, N_CV, "kaggle_resume_structured", random_state=RANDOM_STATE)

gold_raw = pd.concat([workwise_sample, ba_sample, en_sample, cv_sample], ignore_index=True)

print("Gold-Set Größe:", len(gold_raw))
display(gold_raw[["doc_id","source_name","source_type","language","job_title_raw"]].head(25))

Datensätze die fürs sample in Frage kommen (has_both=True): 61818
Gold-Set Größe: 20


,doc_id,source_name,source_type,language,job_title_raw
0,14751-373A61154-S,ba_jobsuche_api_2025,job_ad,de,Schlosser (m/w/d)
1,14751-373A62341-S,ba_jobsuche_api_2025,job_ad,de,Staplerfahrer Schubmast (m/w/d)
2,14751-373A57971-S,ba_jobsuche_api_2025,job_ad,de,Kommissionierer in 3-Schicht (m/w/d)
3,12288-4660333643-S,ba_jobsuche_api_2025,job_ad,de,Sachbearbeiter Logistik (m/w/d)
4,14751-373A58176-S,ba_jobsuche_api_2025,job_ad,de,Industriemechaniker in 3-Schicht (m/w/d)
5,3891071900,kaggle_linkedin_2023_2024_big,job_ad,en,Compensation Analyst
6,3901903592,kaggle_linkedin_2023_2024_big,job_ad,en,Data Analyst
7,3901998039,kaggle_linkedin_2023_2024_big,job_ad,en,Sr. Healthcare Economics Analyst
8,3902860703,kaggle_linkedin_2023_2024_big,job_ad,en,Information System Security Officer
9,3903472063,kaggle_linkedin_2023_2024_big,job_ad,en,Staff Accountant


Of course, n=20 is quite small, so the results merely indicate a trend.

Versioning Goldset IDs:

In [24]:
gold_ids_path = DATA_PROCESSED_EXTERNAL / "goldset_doc_ids.csv"
gold_raw[["doc_id","source_name","source_type","language"]].to_csv(gold_ids_path, index=False)
print("Gespeichert:", gold_ids_path)

Gespeichert: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\goldset_doc_ids.csv


Export as an Excel template for annotation: A file that can be opened in Excel and filled out manually:

In [25]:
OUT_XLSX = DATA_PROCESSED_EXTERNAL / "goldset_for_annotation.xlsx"

gold_export = gold_raw[["doc_id","source_type","source_name","job_title_raw","language","raw_text"]].copy()

gold_export["skills_manual"] = "" # to be filled in manually
gold_export["annotator_notes"] = "" # optional comments

# Excel
gold_export["raw_text"] = gold_export["raw_text"].fillna("").astype(str)
gold_export.to_excel(OUT_XLSX, index=False)
print("Exportiert:", OUT_XLSX)

Exportiert: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\goldset_for_annotation.xlsx


Exporting the Gold Set Template:
- The following fields are exported for each document:
  - `doc_id`, `source_type`, `source_name`
  - `job_title_raw`, `language`
  - `raw_text` (full text of the ad/profile)
  - `skills_manual` (initially empty, to be filled in manually)
  - `annotator_notes` (notes)
- The file `goldset_for_annotation.xlsx` is saved in `data/processed_external/`.
- Annotation rule: Enter a list of skills in `skills_manual` for each entry.

For each document, the skills that actually appear in the text were entered in the `skills_manual` column:
- Skills are extracted directly from the job text
- Spelling remains unchanged (no/minimal normalization)
- Only skills that actually appear
- A document contains approximately 5–20 skills
- Separated by semicolons

## 2. Import the completed Gold Set

Importing the labeled gold set: After manual annotation, the completed file is saved as `goldset_labeled.xlsx`. The `skills_manual` column then contains a semicolon-separated list of skills deemed relevant for each document. The completed table is loaded and serves as the basis for future evaluations: `skills_manual` = true, manually annotated skills; `skills_pred_*` = subsequent lists of automatically extracted skills.

In [26]:
import re

LABELED_XLSX = DATA_PROCESSED_EXTERNAL / "goldset_labeled.xlsx"
df_labeled = pd.read_excel(LABELED_XLSX)

def parse_skill_list(cell): # Display skills as a list/set
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if not s:
        return []
    parts = re.split(r"[;,]", s)  # accept ; or , 
    return [p.strip() for p in parts if p.strip()]

df_labeled["skills_manual_list"] = df_labeled["skills_manual"].apply(parse_skill_list)
df_labeled.head(3)

,doc_id,source_type,source_name,job_title_raw,language,raw_text,skills_manual,annotator_notes,skills_manual_list,n_skills,skills_pred_rulebased,skills_pred_textmining
0,872d2570-cb2c-4325-9edd-f8e45689b9f9,job_ad_scrape,workwise_selenium,Monteur für Sicherheitssysteme (m/w/d),de,Monteur für Sicherheitssysteme (m/w/d) Schnell...,Elektrik; Sicherheitssysteme; Microsoft Office...,NaN,"[Elektrik, Sicherheitssysteme, Microsoft Offic...",9,kundendienst | grubenelektrik | nachrichtentec...,ers | ranch
1,b675769e-cfc1-49bb-8f46-918741adb655,job_ad_scrape,workwise_selenium,Pflegehelfer / Pflegefachassistent (m/w/d),de,Pflegehelfer / Pflegefachassistent (m/w/d) Sch...,Pflegeausbildung; Pflegediagnostik; Pflegeplan...,NaN,"[Pflegeausbildung, Pflegediagnostik, Pflegepla...",13,"qualitätsprüfung, qualitätssicherung | lohn-, ...",ent
2,02222fba-e7f9-4473-af01-56021388985e,job_ad_scrape,workwise_selenium,CAD Datenexperte für 3D Automodelle (m/w/d),de,CAD Datenexperte für 3D Automodelle (m/w/d) Sc...,Qualitätskontrolle; 3D-Dateien; CAD-Dateien; O...,NaN,"[Qualitätskontrolle, 3D-Dateien, CAD-Dateien, ...",19,qualitätsmanagement | clustersysteme | k 0206-...,ste


Quality Control:

In [27]:
df_labeled["n_skills"] = df_labeled["skills_manual"].apply(
    lambda x: len([s for s in str(x).split(";") if s.strip() != ""]))

df_labeled["n_skills"].describe()

count     25.000000
mean      25.920000
std       28.984076
min        9.000000
25%       14.000000
50%       18.000000
75%       23.000000
max      150.000000
Name: n_skills, dtype: float64

A quick quality check for skills_manual: No empty skills, no skills consisting solely of numbers, semicolon separation verified, about 26 skills on average. That’s quite a lot, especially since resumes typically list many skills.

In [28]:
# long-Format, one line per document x Skill
rows = []
for _, row in df_labeled.iterrows():
    for skill in row["skills_manual_list"]:
        rows.append({"doc_id": row["doc_id"],
            "source_name": row["source_name"],
            "skill_manual": skill,
        })

df_labeled_long = pd.DataFrame(rows)
print("Long-Format:", df_labeled_long.shape)
df_labeled.head(10)

Long-Format: (650, 3)


,doc_id,source_type,source_name,job_title_raw,language,raw_text,skills_manual,annotator_notes,skills_manual_list,n_skills,skills_pred_rulebased,skills_pred_textmining
0,872d2570-cb2c-4325-9edd-f8e45689b9f9,job_ad_scrape,workwise_selenium,Monteur für Sicherheitssysteme (m/w/d),de,Monteur für Sicherheitssysteme (m/w/d) Schnell...,Elektrik; Sicherheitssysteme; Microsoft Office...,NaN,"[Elektrik, Sicherheitssysteme, Microsoft Offic...",9,kundendienst | grubenelektrik | nachrichtentec...,ers | ranch
1,b675769e-cfc1-49bb-8f46-918741adb655,job_ad_scrape,workwise_selenium,Pflegehelfer / Pflegefachassistent (m/w/d),de,Pflegehelfer / Pflegefachassistent (m/w/d) Sch...,Pflegeausbildung; Pflegediagnostik; Pflegeplan...,NaN,"[Pflegeausbildung, Pflegediagnostik, Pflegepla...",13,"qualitätsprüfung, qualitätssicherung | lohn-, ...",ent
2,02222fba-e7f9-4473-af01-56021388985e,job_ad_scrape,workwise_selenium,CAD Datenexperte für 3D Automodelle (m/w/d),de,CAD Datenexperte für 3D Automodelle (m/w/d) Sc...,Qualitätskontrolle; 3D-Dateien; CAD-Dateien; O...,NaN,"[Qualitätskontrolle, 3D-Dateien, CAD-Dateien, ...",19,qualitätsmanagement | clustersysteme | k 0206-...,ste
3,20b63a62-eb12-4ebc-b9b2-e329da4656ee,job_ad_scrape,workwise_selenium,Learning Solution Consultant - DACH (m/w/d),de,Learning Solution Consultant - DACH (m/w/d) Sc...,Vertrieb; Consulting; Kundenbetreuung; Projekt...,NaN,"[Vertrieb, Consulting, Kundenbetreuung, Projek...",18,"technisches verständnis | wartung, reparatur, ...",ranch | solution | ste
4,08f24ddb-2d62-408a-b8c1-363cd7c26f6b,job_ad_scrape,workwise_selenium,Bauingenieur (m/w/d),de,Bauingenieur (m/w/d) Schneller-Timer Festanste...,Bauingenieurwesen; Projektabwicklung; Planung;...,NaN,"[Bauingenieurwesen, Projektabwicklung, Planung...",18,arbeitsvorbereitung | produktionsplanung | inb...,ste
5,14751-373A61154-S,job_ad,ba_jobsuche_api_2025,Schlosser (m/w/d),de,Schlosser (m/w/d) Wir suchen ab sofort motivie...,Instandhaltung; Wartung; Reparatur; Betriebsmi...,NaN,"[Instandhaltung, Wartung, Reparatur, Betriebsm...",16,"instandhaltungsmanagement | wartung, reparatur...",alters
6,14751-373A62341-S,job_ad,ba_jobsuche_api_2025,Staplerfahrer Schubmast (m/w/d),de,Staplerfahrer Schubmast (m/w/d) Wir suchen ab ...,Be- und Entladen von LKW; Flurförderfahrzeuge;...,NaN,"[Be- und Entladen von LKW, Flurförderfahrzeuge...",14,"lohn-, gehalts-, tarifwesen | outsourcing | sc...",ent | tas
7,14751-373A57971-S,job_ad,ba_jobsuche_api_2025,Kommissionierer in 3-Schicht (m/w/d),de,Wir suchen zum nächstmöglichen Zeitpunkt Dich ...,Kommissionierung; Versandvorbereitung; Bestell...,NaN,"[Kommissionierung, Versandvorbereitung, Bestel...",9,"einkauf, beschaffung | lohn-, gehalts-, tarifw...",dyna
8,12288-4660333643-S,job_ad,ba_jobsuche_api_2025,Sachbearbeiter Logistik (m/w/d),de,null null Seit mehr als 50 Jahren ist die DIS ...,Verbuchung von Lagerbewegungen; Überwachung vo...,NaN,"[Verbuchung von Lagerbewegungen, Überwachung v...",14,vertriebsmarketing | bedarfsanalyse | vertrieb...,bildung | profi
9,14751-373A58176-S,job_ad,ba_jobsuche_api_2025,Industriemechaniker in 3-Schicht (m/w/d),de,Wir suchen zum nächstmöglichen Zeitpunkt Dich ...,Wartungsarbeiten; Reparaturarbeiten; Montage; ...,NaN,"[Wartungsarbeiten, Reparaturarbeiten, Montage,...",17,technisches verständnis | handwerkliche kenntn...,dyna


Processing of manual labels:
- `skills_manual_list`: Python list of manual skill labels per document.
- Long-format `df_labeled_long` with one row per document×skill combination, e.g., for later analysis or visualization.

**Also add the following to the goldset_labeled Excel file:**
- Column skills_pred_rulebased (separated by |)
- Column skills_pred_textmining (separated by |)

In [29]:
import numpy as np

def _clean_label(x: str) -> str: # Helper, Skill Labels (without URLs, K-codes, prefixes)
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""

    s = str(x).strip()
    if not s:
        return ""

    s = re.sub(r"\b[kK]\s*\d{5,}[\w\-]*\b", "", s) # remove K/k codes (BA)
    s = re.sub(r"^(ESCO|BA|LINKEDIN|RESUME)\s*:\s*", "", s, flags=re.IGNORECASE) # remove prefixes
    s = re.sub(r"https?://\S+", "", s) # remove URLs
    s = re.sub(r"\s*\|\s*", " | ", s) # clean the separators
    s = s.strip(" |()-[]{}") # remove character
    s = re.sub(r"\s+", " ", s).strip() # space
    s = s.lower() # Upper/lowercase

    return s

def _unique_in_order(vals):
    seen = set()
    out = []
    for v in vals:
        if v and v not in seen:
            seen.add(v)
            out.append(v)
    return out

def _aggregate_pred_labels(pred_df: pd.DataFrame, skillid2label: dict) -> pd.Series: # pred_df with: doc_id, skill_id skillid2label maps skill_id, cleaned label/example_label
    if pred_df is None or pred_df.empty:
        return pd.Series(dtype=str)

    tmp = pred_df[["doc_id", "skill_id"]].dropna().copy()
    tmp["doc_id"] = tmp["doc_id"].astype(str).str.strip()
    tmp["skill_id"] = tmp["skill_id"].astype(str).str.strip()
    tmp["label"] = tmp["skill_id"].map(skillid2label).fillna(tmp["skill_id"]) # map skill_id -> example_label, fallbac skill_id
    tmp["label"] = tmp["label"].apply(_clean_label)
    tmp = tmp[tmp["label"].astype(str).str.len() > 0]

    return tmp.groupby("doc_id")["label"].apply(lambda s: " | ".join(_unique_in_order(s.tolist())))

def _build_skillid2label_map(agg_df: pd.DataFrame) -> dict: # Mapping of skill_id to example_label from agg-Parquet
    if agg_df is None or agg_df.empty:
        return {}

    df = agg_df.copy()
    df["skill_id"] = df["skill_id"].astype(str).str.strip()

    if "example_label" not in df.columns: # Ensure that example_label exists
        return {}

    df["example_label"] = df["example_label"].apply(_clean_label)
    df = df.dropna(subset=["example_label"])
    df = df[df["example_label"].astype(str).str.len() > 0]
    df = df.drop_duplicates("skill_id") # Remove duplicates

    return dict(df[["skill_id", "example_label"]].values)

# load labeled goldset
LABELED_XLSX = DATA_PROCESSED_EXTERNAL / "goldset_labeled.xlsx"
df_labeled = pd.read_excel(LABELED_XLSX)
df_labeled["doc_id"] = df_labeled["doc_id"].astype(str).str.strip()
gold_doc_ids = set(df_labeled["doc_id"].dropna().tolist())
print("gold docs:", len(gold_doc_ids))

# Load forecasts (required columns + filters for Gold documents)
print("PRED_FILES keys:", list(PRED_FILES.keys()))

pred_rule = pd.read_parquet(PRED_FILES["rule_based_1.1"], columns=["doc_id", "skill_id"]) # The keys must match
pred_text = pd.read_parquet(PRED_FILES["method_2.1_text_mining"], columns=["doc_id", "skill_id"])
pred_rule["doc_id"] = pred_rule["doc_id"].astype(str).str.strip()
pred_text["doc_id"] = pred_text["doc_id"].astype(str).str.strip()
pred_rule["skill_id"] = pred_rule["skill_id"].astype(str).str.strip()
pred_text["skill_id"] = pred_text["skill_id"].astype(str).str.strip()
pred_rule = pred_rule[pred_rule["doc_id"].isin(gold_doc_ids)]
pred_text = pred_text[pred_text["doc_id"].isin(gold_doc_ids)]
print("pred_rule rows (gold only):", len(pred_rule))
print("pred_text rows (gold only):", len(pred_text))
rule_skill_ids = set(pred_rule["skill_id"].dropna().unique().tolist())
text_skill_ids = set(pred_text["skill_id"].dropna().unique().tolist())
print("unique rule skill_ids:", len(rule_skill_ids))
print("unique text skill_ids:", len(text_skill_ids))

# Label search by method (agg with example_label) + filter to use only skill_ids
RULE_AGG_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skills_rulebased_full.parquet"
TEXT_AGG_PATH = DATA_PROCESSED_EXTERNAL / "method_2_1" / "kldb_skills_2_1_agg.parquet"
# load min columns
df_rule_agg = pd.read_parquet(RULE_AGG_PATH, columns=["skill_id", "example_label"]) if RULE_AGG_PATH.exists() else pd.DataFrame()
df_text_agg = pd.read_parquet(TEXT_AGG_PATH, columns=["skill_id", "example_label"]) if TEXT_AGG_PATH.exists() else pd.DataFrame()
# Filter to show only skills that appear in Gold predictions (caches data)
if not df_rule_agg.empty:
    df_rule_agg["skill_id"] = df_rule_agg["skill_id"].astype(str).str.strip()
    df_rule_agg = df_rule_agg[df_rule_agg["skill_id"].isin(rule_skill_ids)]

if not df_text_agg.empty:
    df_text_agg["skill_id"] = df_text_agg["skill_id"].astype(str).str.strip()
    df_text_agg = df_text_agg[df_text_agg["skill_id"].isin(text_skill_ids)]

rule_map = _build_skillid2label_map(df_rule_agg)
text_map = _build_skillid2label_map(df_text_agg)

print("rule_map size:", len(rule_map), "| text_map size:", len(text_map))

# Aggregate + write to Excel
rule_series = _aggregate_pred_labels(pred_rule, rule_map)
text_series = _aggregate_pred_labels(pred_text, text_map)
df_labeled["skills_pred_rulebased"] = df_labeled["doc_id"].map(rule_series).fillna("")
df_labeled["skills_pred_textmining"] = df_labeled["doc_id"].map(text_series).fillna("")
df_labeled.to_excel(LABELED_XLSX, index=False)
print("Saved:", LABELED_XLSX)

display(df_labeled[["doc_id", "skills_manual", "skills_pred_rulebased", "skills_pred_textmining"]].head(25))

gold docs: 25
PRED_FILES keys: ['rule_based_1.1', 'method_2.1_text_mining']
pred_rule rows (gold only): 989
pred_text rows (gold only): 56
unique rule skill_ids: 663
unique text skill_ids: 49
rule_map size: 653 | text_map size: 43
Saved: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\goldset_labeled.xlsx


,doc_id,skills_manual,skills_pred_rulebased,skills_pred_textmining
0,872d2570-cb2c-4325-9edd-f8e45689b9f9,Elektrik; Sicherheitssysteme; Microsoft Office...,kundendienst | grubenelektrik | nachrichtentec...,ers | ranch
1,b675769e-cfc1-49bb-8f46-918741adb655,Pflegeausbildung; Pflegediagnostik; Pflegeplan...,"qualitätsprüfung, qualitätssicherung | lohn-, ...",ent
2,02222fba-e7f9-4473-af01-56021388985e,Qualitätskontrolle; 3D-Dateien; CAD-Dateien; O...,qualitätsmanagement | clustersysteme | k 0206-...,ste
3,20b63a62-eb12-4ebc-b9b2-e329da4656ee,Vertrieb; Consulting; Kundenbetreuung; Projekt...,"technisches verständnis | wartung, reparatur, ...",ranch | solution | ste
4,08f24ddb-2d62-408a-b8c1-363cd7c26f6b,Bauingenieurwesen; Projektabwicklung; Planung;...,arbeitsvorbereitung | produktionsplanung | inb...,ste
5,14751-373A61154-S,Instandhaltung; Wartung; Reparatur; Betriebsmi...,"instandhaltungsmanagement | wartung, reparatur...",alters
6,14751-373A62341-S,Be- und Entladen von LKW; Flurförderfahrzeuge;...,"lohn-, gehalts-, tarifwesen | outsourcing | sc...",ent | tas
7,14751-373A57971-S,Kommissionierung; Versandvorbereitung; Bestell...,"einkauf, beschaffung | lohn-, gehalts-, tarifw...",dyna
8,12288-4660333643-S,Verbuchung von Lagerbewegungen; Überwachung vo...,vertriebsmarketing | bedarfsanalyse | vertrieb...,bildung | profi
9,14751-373A58176-S,Wartungsarbeiten; Reparaturarbeiten; Montage; ...,technisches verständnis | handwerkliche kenntn...,dyna


**Observation:**

The “skills” output by Method 2.1 (text mining) in the first ten lines appear largely implausible. There is a high frequency of very short tokens or generic words that look more like misclassifications than actual skills. It is striking that this pattern is particularly visible in the first ten entries, while the subsequent examples appear more plausible overall. This is due to the fact that the first ten documents are German-language sources (Workwise scraping and BA Jobsuche API), whereas the majority of the remaining examples come from English-language datasets. From this, it can be inferred that Method 2.1 performs significantly worse for German documents in the current setup. A plausible cause is the highly unbalanced dataset. There are only approximately 4,000 German documents compared to approximately 190,000 documents in total (predominantly English). Since the method relies on frequencies/co-occurrences, it is less able to identify robust, “skill-like” terms in the small German subset and instead tends to fall back on nonspecific tokens. Even when randomly selecting other German datasets for the evaluation set, a largely similar picture emerges, with no significant improvement.

This effect is likely to have a correspondingly negative impact on the evaluation metrics of Method 2.1, particularly in the DE examples (e.g., due to low recall and/or many false positives).
The reason for this is that TF-IDF primarily identifies statistically salient terms that are not necessarily semantically interpretable as “skills,” and the small DE subset thus generates more unstable candidates (Karakatsanis et al., 2017; Pejic-Bach et al., 2020). Additionally, preprocessing/formatting artifacts and string similarity mapping can lead to false positives. As a result, the quality of the 2.1 predictions on DE data declines.

**Note:**  
To investigate potential language effects, a separate notebook (08b in the subfolder of 08) was additionally created with a gold set consisting solely of English text. The results are evaluated comparatively there.

In [30]:
# Check, the two pred lists
print("pred_rule columns:", pred_rule.columns.tolist())
print("pred_text columns:", pred_text.columns.tolist())
display(pred_rule.head(3))
display(pred_text.head(3))

pred_rule columns: ['doc_id', 'skill_id']
pred_text columns: ['doc_id', 'skill_id']


,doc_id,skill_id
1166474,3891071900,BA:K 030205-010
1166475,3891071900,BA:K 030206-004
1166476,3891071900,BA:K 030300-025


,doc_id,skill_id
11866,3891071900,RESUME:provider analysis
11867,3891071900,RESUME:appropriate solutions
13916,3894284559,LINKEDIN:mutual funds


## 3. Evaluation of Skill Extraction Against the Gold Set

- Gold: skills_manual/skills_manual_list -> Gold Label Set
- For each profile extension method, a `skills_pred_list` column should be generated for skill extraction. (Pred: skills_pred_rulebased and skills_pred_textmining (separated) -> Pred Label Set)
- Both sides use the same normalization
- The function `eval(...)` calculates metrics on a per-document basis (TP/FP/FN, Precision/Recall/F1 per Doc, plus Macro and Micro) and returns macro averages.
- The setup is based on the evaluation methods used in skill extraction and NER studies (e.g., Gugnani et al. 2020; Cenikj et al. 2021; Decorte et al. 2023), which use precision/recall/F1 or ranking metrics.
- Relaxed Matching: with a similarity threshold to account for minor spelling variations

### 3.1 Normalization: Comparable, Human-Readable Skill Strings

Normalization ensures that strings are comparable (whitespace, prefixes) so that matching does not fail due to formatting artifacts.
- Removes prefixes (ESCO:/BA:/LINKEDIN:/RESUME:)
- Removes K/k codes at the beginning of BA skills
- Removes URLs (esco)
- Normalizes whitespace, lowercase
- Leaves special characters such as + # / & . - in place (e.g., for tech skills, etc.)

In [31]:
from difflib import SequenceMatcher

_ALLOWED_CHARS = re.compile(r"[^\w\s\.\-\+\#\/&'()\u00C0-\u017F]", flags=re.UNICODE) # Allowed: Unicode letters + numbers + whitespace + . - + # / & ' ( )

def normalize_label(s: str) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip()
    if not s:
        return ""

    s = re.sub(r"https?://\S+", "", s).strip() # Remove URLs
    s = re.sub(r"^(ESCO|BA|LINKEDIN|RESUME)\s*:\s*", "", s, flags=re.IGNORECASE).strip() # Remove any prefixes that are still there
    s = re.sub(r"^[kK]\s*\d{3,}[\w\-]*\s*[-:]?\s*", "", s).strip() # K-Codes by BA
    s = s.strip(" |;,:-–—()[]{}") # Separator/Noise at the edge
    s = s.replace("’", "'").replace("–", "-").replace("—", "-") # Reduce special characters, but keep the important ones
    s = _ALLOWED_CHARS.sub("", s) # Special characters
    s = re.sub(r"\s+", " ", s).strip() # Normalize whitespace
    s = s.lower() # Technical skills
    s = s.replace("c + +", "c++").replace("c #", "c#")
    s = s.replace("ci / cd", "ci/cd")

    return s

### 3.2 Creating lists from Excel columns:

In [32]:
def split_pipe(cell) -> list[str]: # Predictions a | b | c as list: list[str]
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    s = str(cell).strip()
    if not s:
        return []
    parts = [p.strip() for p in s.split("|")]
    return [p for p in parts if p]

def split_manual(cell) -> list[str]: # skills_manual Semicolon-separated; accepts ;, |, and line breaks as delimiters
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    s = str(cell).strip()
    if not s:
        return []
    s = s.strip().strip("[]") # Remove the square brackets
    parts = re.split(r"[;,\|\n]+", s)
    parts = [p.strip() for p in parts]
    return [p for p in parts if p]

### 3.3 Relaxed Matching

- By default, without Relaxed Match, the system uses exact matching (strict 1-to-1 matching). Matching is based on the same procedure as in the two notebooks for Profile Extension 10b & 11
- Relaxed Matching ensures a less strict matching process here, so that the results of this test are more meaningful and yield better results. Since there are comparatively very few external datasets and skills, very poor results are to be expected without a less strict matching process. This means that, for example, minor typos or slightly different spellings, etc., do not automatically result in missed matches.
- It accepts the closest Pred-Skill per Gold-Skill if the similarity is ≥ threshold (greedy 1:1 matching to avoid double counting).
- Test here with multiple thresholds (RELAXED_THRESHOLD = default, 0.95, 0.92, and 0.90). The higher the value, the stricter the matching.
- Relaxed matching can artificially improve metrics if the similarity threshold is set too low. Therefore, it is tested transparently and safeguarded with a check in 3.5.

In [33]:
USE_RELAXED_MATCH = True # False -> default, True for final evaluation
RELAXED_THRESHOLD = 0.90 # The higher the value, the stricter the settings; tested with default, 0.95, 0.92, and 0.90; 0.90 is used here as the main option

def is_close(a: str, b: str) -> bool:
    return SequenceMatcher(None, a, b).ratio() >= RELAXED_THRESHOLD

def match_sets(gold: set[str], pred: set[str]) -> tuple[set[str], set[str], set[str]]: # Returns: tp, fp, fn; exact match (1-to-1 match) if USE_RELAXED_MATCH=False, otherwise greedy relaxed matching (gold vs. pred) based on high similarity
    if not USE_RELAXED_MATCH:
        tp = gold & pred
        fp = pred - gold
        fn = gold - pred
        return tp, fp, fn

    pred_left = set(pred) # relaxed greedy, to prevent duplicate matches
    tp = set()

    # first exact
    exact = gold & pred_left
    tp |= exact
    pred_left -= exact

    # then relaxed for other labels
    for g in gold:
        if g in pred_left:
            tp.add(g)
            pred_left.remove(g)
            continue
        hit = None # trying to find a similar post
        for p in pred_left:
            if is_close(g, p):
                hit = p
                break
        if hit is not None:
            tp.add(g)  # gold as a hit
            pred_left.remove(hit)

    fp = pred_left
    fn = gold - tp
    return tp, fp, fn

### 3.4 Evaluation

Evaluation per document + Macro/Micro + Debug tables, using the current goldset_labeled.xlsx file with manually annotated skills and the two “Pred” columns:
- Per document: TP/FP/FN -> Precision/Recall/F1
- Macro: Average across documents
- Micro: All together (sum of TP/FP/FN)

**Interpretation:** For each document, compare the manually annotated gold set (`skills_manual`) with the predicted skill sets from the methods.

Definitions per document:
- TP (True Positives): Skills that are included in the gold set and were also predicted.
- FP (False Positives): Skills that were predicted but are not in the gold set.
- FN (False Negatives): Skills that are in the gold set but were not predicted.

Calculate from this:
- Precision = TP/(TP+FP) -> How accurate are the predictions? (few false positives)
- Recall = TP/(TP+FN) -> How completely are gold skills found
- F1 = harmonic mean of Precision & Recall -> balance metric

Macro & Micro:
- Macro: Average of Precision/Recall/F1 across documents (each document counts equally).
- Micro: Sums TP/FP/FN across all documents and calculates an overall Precision/Recall/F1 -> larger documents with many skills have a stronger influence (e.g., longer CVs).

(Source: Gugnani et al. 2020; Jayaswal, 2020, https://towardsdatascience.com/performance-metrics-confusion-matrix-precision-recall-and-f1-score-a8fe076a2262/)

In [34]:
LABELED_XLSX = DATA_PROCESSED_EXTERNAL / "goldset_labeled.xlsx" # load Gold-Excel
df_labeled = pd.read_excel(LABELED_XLSX)

df_labeled["doc_id"] = df_labeled["doc_id"].astype(str).str.strip() # doc_id as string 
need_cols = ["doc_id", "skills_manual", "skills_pred_rulebased", "skills_pred_textmining"] # Columns

df_labeled["gold_list_raw"] = df_labeled["skills_manual"].apply(split_manual) # Build lists
df_labeled["pred_rule_list_raw"] = df_labeled["skills_pred_rulebased"].apply(split_pipe)
df_labeled["pred_text_list_raw"] = df_labeled["skills_pred_textmining"].apply(split_pipe)

df_labeled["gold_set"] = df_labeled["gold_list_raw"].apply(lambda lst: set([normalize_label(x) for x in lst if normalize_label(x)])) # Normalize + Sets
df_labeled["pred_rule_set"] = df_labeled["pred_rule_list_raw"].apply(lambda lst: set([normalize_label(x) for x in lst if normalize_label(x)]))
df_labeled["pred_text_set"] = df_labeled["pred_text_list_raw"].apply(lambda lst: set([normalize_label(x) for x in lst if normalize_label(x)]))

def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2*p*r/(p+r)) if (p+r) else 0.0
    return p, r, f1

def eval_one(row, pred_col: str):
    gold = row["gold_set"]
    pred = row[pred_col]
    tp_set, fp_set, fn_set = match_sets(gold, pred)
    tp, fp, fn = len(tp_set), len(fp_set), len(fn_set)
    p, r, f1 = prf(tp, fp, fn)
    return pd.Series({
        "tp": tp, "fp": fp, "fn": fn,
        "precision": p, "recall": r, "f1": f1,
        "tp_list": sorted(tp_set),
        "fp_list": sorted(fp_set),
        "fn_list": sorted(fn_set),
        "n_gold": len(gold),
        "n_pred": len(pred),
    })

# Evaluate by method
eval_rule = df_labeled.apply(lambda row: eval_one(row, "pred_rule_set"), axis=1)
eval_text = df_labeled.apply(lambda row: eval_one(row, "pred_text_set"), axis=1)

df_rule = pd.concat([df_labeled[["doc_id","source_name","source_type","language","job_title_raw"]], eval_rule], axis=1)
df_rule["method"] = "rule_based_1.1"
df_text = pd.concat([df_labeled[["doc_id","source_name","source_type","language","job_title_raw"]], eval_text], axis=1)
df_text["method"] = "method_2_1_text_mining"
df_eval = pd.concat([df_rule, df_text], ignore_index=True)

# Macro
macro = df_eval.groupby("method")[["precision","recall","f1"]].mean().sort_values("f1", ascending=False)
print("MACRO (Durchschnitt pro Dokument):")
display(macro)

# Micro
micro_rows = []
for m, g in df_eval.groupby("method"):
    tp, fp, fn = int(g["tp"].sum()), int(g["fp"].sum()), int(g["fn"].sum())
    p, r, f1 = prf(tp, fp, fn)
    micro_rows.append({"method": m, "tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": f1})
df_micro = pd.DataFrame(micro_rows).sort_values("f1", ascending=False)

print("MICRO (alles zusammen, Set-basiert):")
display(df_micro)

MACRO (Durchschnitt pro Dokument):


,precision,recall,f1
method,,,
rule_based_1.1,0.120489,0.176179,0.134247
method_2_1_text_mining,0.093333,0.007607,0.013774


MICRO (alles zusammen, Set-basiert):


,method,tp,fp,fn,precision,recall,f1
1,rule_based_1.1,152,831,498,0.154629,0.233846,0.186160
0,method_2_1_text_mining,9,45,641,0.166667,0.013846,0.025568


**Comparison of the Thresholds Tested**

Three variants were tested in this notebook (Micro/Macro for both methods):

1) Exact Match (Relaxed = False)
    - rule_based_1.1: Macro-F1 ≈ 0.127, Micro-F1 ≈ 0.182
    - method_2_1_text_mining: Macro-F1 ≈ 0.011, Micro-F1 ≈ 0.017

2) Relaxed Threshold = 0.95
    - rule_based_1.1: Macro-F1 ≈ 0.134, Micro-F1 ≈ 0.186  
    - method_2_1_text_mining: Macro-F1 ≈ 0.014, Micro-F1 ≈ 0.026

3) Relaxed Threshold = 0.90  
    - rule_based_1.1: Macro-F1 ≈ 0.134, Micro-F1 ≈ 0.186  
    - method_2_1_text_mining: Macro-F1 ≈ 0.014, Micro-F1 ≈ 0.026

4) Additionally, a compromise was tested: 0.92 slightly improves text mining, but yields less gain than 0.90 and barely improves rule-based.

**Decision:**

For the final evaluation, we use Relaxed Matching with Threshold = 0.90 because:
- the results (especially Micro-F1 and Recall) consistently improve,
- without Precision dropping significantly,
- and the data contains realistic spelling variations.

**Interpretation:**

The absolute F1 scores are low overall. This is partly because the gold set (especially CVs) contains a very large number of skills per document, making recall difficult to achieve, while the methods have very different output densities:  
- rule_based_1.1 generates many skill candidates -> higher recall (23.5% of the gold skills were found), but many false positives -> limited precision
- method_2_1_text_mining generates very few skills per document -> precision may be acceptable, but recall is very low (1.4%), causing F1 to drop sharply.
- The fact that Micro outperforms Macro (rule_based) suggests that documents with many skills have a stronger influence on the overall score and yield more true positives in absolute terms.
- Although rule_based is better, it still performs weakly, with many FPs (e.g., because the vocabulary is broad/dictionary matching generates many candidates)

**Supplementary Tests** using a purely English-language gold set (Notebook 08b) actually show very similar metrics or only marginally better results. The observed effects are therefore not attributable to the language distribution in the gold set (at least not to a decisive extent).

Error analysis based on documents and sources:

In [35]:
# the largest gaps/the most false positives
print("Top-5 Docs mit meisten False Negatives (FN) je Methode:")
for m in df_eval["method"].unique():
    display(df_eval[df_eval["method"] == m].sort_values("fn", ascending=False).head(5)[["doc_id","source_name","language","job_title_raw","n_gold","n_pred","tp","fp","fn","fn_list"]])

print("Top-5 Docs mit meisten False Positives (FP) je Methode:")
for m in df_eval["method"].unique():
    display(df_eval[df_eval["method"] == m].sort_values("fp", ascending=False).head(5)[["doc_id","source_name","language","job_title_raw","n_gold","n_pred","tp","fp","fn","fp_list"]])

Top-5 Docs mit meisten False Negatives (FN) je Methode:


,doc_id,source_name,language,job_title_raw,n_gold,n_pred,tp,fp,fn,fn_list
23,resume_struct_42378,kaggle_resume_structured,en,Front End Developer,150,108,45,63,105,"[accessibility testing, adobe dreamweaver, ado..."
22,resume_struct_31072,kaggle_resume_structured,en,Sr. Java Full Stack developer,73,56,24,32,49,"[angular 2, angular 4, angular js, boss, cvs, ..."
21,resume_struct_47135,kaggle_resume_structured,en,IT Security Analyst,40,48,15,33,25,"[access/identity management, active directory,..."
13,3902860703,kaggle_linkedin_2023_2024_big,en,Information System Security Officer,23,47,2,45,21,"[asset and vulnerability management, casp+ ce,..."
18,3894284559,kaggle_linkedin_2023_2024_big,en,529 Shareholder Services Specialist II,25,49,4,45,21,"[account maintenance updates, analytical skill..."


,doc_id,source_name,language,job_title_raw,n_gold,n_pred,tp,fp,fn,fn_list
48,resume_struct_42378,kaggle_resume_structured,en,Front End Developer,150,8,4,4,146,"[accessibility, accessibility testing, adobe d..."
47,resume_struct_31072,kaggle_resume_structured,en,Sr. Java Full Stack developer,73,4,2,2,71,"[agile, angular, angular 2, angular 4, angular..."
46,resume_struct_47135,kaggle_resume_structured,en,IT Security Analyst,40,2,1,1,39,"[access/identity management, active directory,..."
49,resume_struct_4696,kaggle_resume_structured,en,Senior Full Stack Developer,36,3,1,2,35,"[agile, angularjs, aws, bootstrap, bower, css3..."
43,3894284559,kaggle_linkedin_2023_2024_big,en,529 Shareholder Services Specialist II,25,3,0,3,25,"[account maintenance updates, adobe acrobat, a..."


Top-5 Docs mit meisten False Positives (FP) je Methode:


,doc_id,source_name,language,job_title_raw,n_gold,n_pred,tp,fp,fn,fp_list
12,3901998039,kaggle_linkedin_2023_2024_big,en,Sr. Healthcare Economics Analyst,14,85,4,81,10,"[accounting, affirmative action, application, ..."
23,resume_struct_42378,kaggle_resume_structured,en,Front End Developer,150,108,45,63,105,"[a/b-testing, adobe, agile, angular, benchmark..."
3,20b63a62-eb12-4ebc-b9b2-e329da4656ee,workwise_selenium,de,Learning Solution Consultant - DACH (m/w/d),18,53,3,50,15,"[abteilungs- bereichs- ressortleitung, arbeits..."
2,02222fba-e7f9-4473-af01-56021388985e,workwise_selenium,de,CAD Datenexperte für 3D Automodelle (m/w/d),19,51,2,49,17,"[administrative, adobe muse, benutzerschnittst..."
15,3906225987,kaggle_linkedin_2023_2024_big,en,Project Coordinator,20,54,7,47,13,"[administrative, application process, cash flo..."


,doc_id,source_name,language,job_title_raw,n_gold,n_pred,tp,fp,fn,fp_list
41,3903878153,kaggle_linkedin_2023_2024_big,en,Dentist,17,5,0,5,17,"[assisted living, daily, direct, require, ski]"
48,resume_struct_42378,kaggle_resume_structured,en,Front End Developer,150,8,4,4,146,"[cross browser, cross-platform compatibility, ..."
43,3894284559,kaggle_linkedin_2023_2024_big,en,529 Shareholder Services Specialist II,25,3,0,3,25,"[daily, financial, mutual funds]"
28,20b63a62-eb12-4ebc-b9b2-e329da4656ee,workwise_selenium,de,Learning Solution Consultant - DACH (m/w/d),18,3,0,3,18,"[ranch, solution, ste]"
25,872d2570-cb2c-4325-9edd-f8e45689b9f9,workwise_selenium,de,Monteur für Sicherheitssysteme (m/w/d),9,2,0,2,9,"[ers, ranch]"


In [36]:
# broken down by source (source_name)
print("Aufgelistet nach source_name (Macro):")
by_source = df_eval.groupby(["method","source_name"])[["precision","recall","f1"]].mean().sort_values(["method","f1"], ascending=[True, False])
display(by_source)

Aufgelistet nach source_name (Macro):


precision    recall  \
method                 source_name                                          
method_2_1_text_mining kaggle_resume_structured        0.366667  0.021368   
                       kaggle_linkedin_2023_2024_big   0.050000  0.008333   
                       ba_jobsuche_api_2025            0.000000  0.000000   
                       workwise_selenium               0.000000  0.000000   
rule_based_1.1         kaggle_resume_structured        0.349643  0.321642   
                       kaggle_linkedin_2023_2024_big   0.087145  0.200518   
                       workwise_selenium               0.041490  0.109942   
                       ba_jobsuche_api_2025            0.037022  0.048273   

                                                            f1  
method                 source_name                              
method_2_1_text_mining kaggle_resume_structured       0.040296  
                       kaggle_linkedin_2023_2024_big  0.014286  
                       ba_jobsuche_api_2025           0.000000  
                       workwise_selenium              0.000000  
rule_based_1.1         kaggle_resume_structured       0.331361  
                       kaggle_linkedin_2023_2024_big  0.119022  
                       workwise_selenium              0.060098  
                       ba_jobsuche_api_2025           0.041732

The Top-FN/Top-FP lists show that errors vary significantly depending on document type: CV documents contain a particularly large number of skills (Gold), which causes methods with few predictions (text mining) to systematically produce many false negatives. Structured CVs, on the other hand, yield the best results with both methods; this can be attributed, for example, to the IT focus of this dataset as well as the broad coverage of the skill vocabularies in this field.
The evaluation broken down by `source_name` empirically shows clear differences between sources (CV vs. job ads, DE vs. EN, scraped data vs. Kaggle). This suggests that the overall values should not be interpreted as global performance but rather viewed on a domain-specific basis. They differ, for example, in text length, style, skill density, and terminology, which directly affects precision and recall.

### 3.5 Debug/Check: Exact vs. Relaxed

Additionally, a debug/check is used to verify the newly found matches (Relaxed-only) and ensure that they appear reasonable. Per document:
- how many hits with Exact
- how many hits with Relaxed
- which newly added pairs (gold <-> pred (score))

In [37]:
DEBUG_THRESHOLD = 0.90  # Here you can compare version 0.90 and 0.92

def relaxed_match_pairs(gold_set: set[str], pred_set: set[str], threshold: float): # greedy 1:1 vs. relaxed Übereinstimmung, gibt zurück: matched_gold, matched_pred, Paare [(g, p, score)] für ausschließlich relaxed Matches
    gold = set(gold_set)
    pred_left = set(pred_set)

    # first exact
    exact = gold & pred_left
    pred_left -= exact
    gold_left = list(gold - exact)
    pairs = []
    matched_gold = set(exact)
    matched_pred = set(exact)

    for g in gold_left:
        best = None
        best_score = 0.0
        for p in pred_left:
            sc = SequenceMatcher(None, g, p).ratio()
            if sc >= threshold and sc > best_score:
                best = p
                best_score = sc
        if best is not None:
            matched_gold.add(g)
            matched_pred.add(best)
            pred_left.remove(best)
            pairs.append((g, best, best_score))

    return matched_gold, matched_pred, pairs

debug_rows = []

for _, r in df_labeled.iterrows():
    doc_id = str(r["doc_id"])
    gold = r["gold_set"]

    for method, pred_col in [("rule_based_1.1", "pred_rule_set"), ("method_2_1_text_mining", "pred_text_set")]:
        pred = r[pred_col]
        tp_exact = gold & pred # exact tps
        matched_gold, matched_pred, pairs = relaxed_match_pairs(gold, pred, threshold=DEBUG_THRESHOLD) # relaxed matches
        gained_gold = matched_gold - tp_exact  # Gold-Labels new counted as TP
        gained_pairs = [(g, p, sc) for (g, p, sc) in pairs if g in gained_gold] # Show only pairs that match the matches won

        debug_rows.append({
            "doc_id": doc_id,
            "method": method,
            "n_gold": len(gold),
            "n_pred": len(pred),
            "tp_exact": len(tp_exact),
            "tp_relaxed": len(matched_gold),
            "gained_tp": len(gained_gold),
            "gained_pairs_top": " | ".join([f"{g} ↔ {p} ({sc:.2f})" for g,p,sc in sorted(gained_pairs, key=lambda x: -x[2])[:10]])})

df_debug = pd.DataFrame(debug_rows)
display(df_debug.sort_values(["gained_tp", "tp_relaxed"], ascending=False).head(30))
print("Docs mit gained_tp > 0:", int((df_debug["gained_tp"] > 0).sum()), "von", len(df_debug))

display(df_debug.groupby("method")[["tp_exact","tp_relaxed","gained_tp"]].sum()) # Totals by method

,doc_id,method,n_gold,n_pred,tp_exact,tp_relaxed,gained_tp,gained_pairs_top
45,resume_struct_31072,method_2_1_text_mining,73,4,0,2,2,multi-threading ↔ multi threading (0.93) | gro...
47,resume_struct_42378,method_2_1_text_mining,150,8,3,4,1,html/html5 ↔ html html5 (0.90)
26,3902860703,rule_based_1.1,23,47,1,2,1,risk management framework (rmf ↔ risk manageme...
12,14751-373A62341-S,rule_based_1.1,14,19,0,1,1,kommissionierung ↔ kommissionieren (0.90)
14,14751-373A57971-S,rule_based_1.1,9,13,0,1,1,kommissionierung ↔ kommissionieren (0.90)
46,resume_struct_42378,rule_based_1.1,150,108,45,45,0,
44,resume_struct_31072,rule_based_1.1,73,56,24,24,0,
48,resume_struct_4696,rule_based_1.1,36,40,16,16,0,
42,resume_struct_47135,rule_based_1.1,40,48,15,15,0,
30,3906225987,rule_based_1.1,20,54,7,7,0,


Docs mit gained_tp > 0: 5 von 50


,tp_exact,tp_relaxed,gained_tp
method,,,
method_2_1_text_mining,6,9,3
rule_based_1.1,149,152,3


The debug comparison shows that relaxed matching generates meaningful additional true positives only in a few cases (5 out of 50 document-method combinations with `gained_tp > 0`). Since the overall gain in TP remains low, the low overall metrics cannot be explained primarily by spelling variations, but rather by differences in content (missing skills, different granularity, or varying skill formulations).

# Conclusion: Notebook 08

This notebook creates a small gold/evaluation dataset and provides a complete evaluation procedure to fairly compare different skill extraction methods on the same documents.  

The evaluation reveals clear differences in the output:
- The rule-based method currently achieves higher recall (while generating many false positives), whereas text mining outputs only very few skills per document, resulting in extremely low recall.
- Dictionary/taxonomy-based methods tend to robustly identify “classic” skills, while text mining without strong NER/disambiguation quickly misses a lot or extracts only a few highly precise skills.
- The comparison (Exact vs. Relaxed Matching) shows that spelling variations alone explain only a small portion of the discrepancies; the main gaps are content-related (missing skills, granularity, synonyms, or the issue with the German datasets). Supplementary tests with a purely English-language gold set (Notebook 08b) show very similar metrics. The observed effects are therefore not attributable to the language distribution in the gold set.

**Overall assessment:**

The overall low metrics should be interpreted in light of the very small, heterogeneous gold set (n = 25) as well as the two methods, which were deliberately kept simple. Both approaches are based on rule- or vocabulary-driven methods and are primarily designed for scalability across massive, unstructured datasets, not for maximum precision in individual documents. Higher performance values are expected for more complex methods such as transformer- or LLM-based skill extraction; however, these require significantly more computational and modeling resources and lie outside the methodological focus of this work (e.g., Clavié & Soulié, 2023; Decorte et al., 2023; Gugnani et al., 2020; Li et al., 2023; Rosenberger et al., 2025).